In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

def build_and_train_mnist_cnn():
    """
    Builds, trains, evaluates a CNN on the MNIST dataset,
    and visualizes 5 sample predictions.
    """

    print(f"Using TensorFlow version: {tf.__version__}")

    # --- 1. Load and Preprocess Data ---

    # Load the MNIST dataset
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

    # Store original test labels for visualization
    y_test_orig = y_test

    # Preprocess the images
    # Reshape to (num_samples, 28, 28, 1) to include the channel dimension
    # Normalize pixel values from [0, 255] to [0.0, 1.0]
    x_train = x_train.astype("float32").reshape(-1, 28, 28, 1) / 255.0
    x_test = x_test.astype("float32").reshape(-1, 28, 28, 1) / 255.0

    print(f"x_train shape: {x_train.shape}")
    print(f"x_test shape: {x_test.shape}")

    # Preprocess the labels
    # Convert labels to one-hot encoded vectors
    # e.g., 5 -> [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
    num_classes = 10
    y_train = keras.utils.to_categorical(y_train, num_classes)
    y_test = keras.utils.to_categorical(y_test, num_classes)

    print(f"y_train shape (one-hot): {y_train.shape}")
    print(f"y_test shape (one-hot): {y_test.shape}")

    # --- 2. Build the CNN Model ---

    # Define the model architecture using the Keras Sequential API
    model = keras.Sequential(
        [
            # Input layer
            keras.Input(shape=(28, 28, 1)),

            # Convolutional Layer 1
            # 32 filters, 3x3 kernel size, 'relu' activation
            layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),

            # Max Pooling Layer 1
            # 2x2 pool size
            layers.MaxPooling2D(pool_size=(2, 2)),

            # Convolutional Layer 2
            # 64 filters, 3x3 kernel
            layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),

            # Max Pooling Layer 2
            layers.MaxPooling2D(pool_size=(2, 2)),

            # Flatten the 3D feature maps to 1D vectors
            layers.Flatten(),

            # Add a Dropout layer to prevent overfitting
            # Randomly drops 50% of units during training
            layers.Dropout(0.5),

            # Fully Connected (Dense) Layer
            # 128 units, 'relu' activation
            layers.Dense(128, activation="relu"),

            # Output Layer
            # 10 units (one for each class), 'softmax' activation
            # Softmax provides a probability distribution over the classes
            layers.Dense(num_classes, activation="softmax"),
        ]
    )

    # Display the model's architecture
    model.summary()

    # --- 3. Compile and Train the Model ---

    # Compile the model, specifying the optimizer, loss function, and metrics
    model.compile(
        loss="categorical_crossentropy",  # Use for one-hot encoded labels
        optimizer="adam",                 # A popular and effective optimizer
        metrics=["accuracy"],             # Track accuracy during training
    )

    # Set training parameters
    batch_size = 128
    epochs = 10

    print("\n--- Starting Model Training ---")

    # Train the model
    history = model.fit(
        x_train,
        y_train,
        batch_size=batch_size,
        epochs=epochs,
        validation_split=0.1,  # Use 10% of training data for validation
        verbose=1,
    )

    print("--- Model Training Finished ---")

    # --- 4. Evaluate the Model ---

    print("\n--- Evaluating Model on Test Data ---")

    # Evaluate the trained model on the test dataset
    score = model.evaluate(x_test, y_test, verbose=0)

    print(f"Test loss: {score[0]:.4f}")
    print(f"Test accuracy: {score[1]:.4f} ({(score[1] * 100):.2f}%)")

    # Check if the goal is met
    if score[1] > 0.95:
        print("Goal achieved: Test accuracy is > 95%!")
    else:
        print("Goal not met: Test accuracy is <= 95%.")

    # --- 5. Visualize Predictions ---

    print("\n--- Visualizing Sample Predictions ---")

    # Get model's predictions on the test set
    # predictions will be arrays of 10 probabilities (from softmax)
    predictions = model.predict(x_test)

    # Get the class with the highest probability
    # e.g., [0.1, 0.0, ..., 0.8, ...] -> 7
    predicted_labels = np.argmax(predictions, axis=1)

    # Select 5 random indices from the test set
    num_samples = 5
    sample_indices = np.random.choice(x_test.shape[0], num_samples, replace=False)

    # Create a figure to plot the images
    plt.figure(figsize=(15, 5))

    for i, idx in enumerate(sample_indices):
        # Create a subplot for each image
        ax = plt.subplot(1, num_samples, i + 1)

        # Reshape the image from (28, 28, 1) to (28, 28) for plotting
        img = x_test[idx].reshape(28, 28)

        # Plot the image
        plt.imshow(img, cmap="gray")

        # Set the title
        true_label = y_test_orig[idx]
        pred_label = predicted_labels[idx]

        title = f"True: {true_label}\nPred: {pred_label}"

        # Color the title green for correct, red for incorrect
        if pred_label == true_label:
            ax.set_title(title, color="green")
        else:
            ax.set_title(title, color="red")

        # Remove axes
        plt.axis("off")

    # Show the plot
    plt.tight_layout()
    plt.show()
    print(f"Displayed {num_samples} sample predictions. Check the plot window.")


if __name__ == "__main__":

    build_and_train_mnist_cnn()